# Z-profile spot intensity — per-z diagnostic for one (hyb round, FOV)

Deep-dive counterpart to `fast_spot_quantification.ipynb`: that notebook does
a *fast* scan across many FOVs by max-projecting a handful of sampled
z-planes per (round, color). This notebook instead does a *slow, thorough*
look at **one** hyb round + FOV: it detects foci **independently on every
real z-plane** (no projection, no sampling) for each color channel, so a
human can judge whether a detected focus's intensity-vs-z profile looks like
a genuine 3-D punctum or noise/dirt.

Can also load the raw image from **outside** the normal
`round_info.csv`-driven experiment structure (e.g. an old/duplicate hyb
folder that isn't one of the directories `round_info.csv` declares) — see
`CUSTOM_IMAGE_PATH`/`CUSTOM_IMAGE_DIR` in the Parameters section below.

## 1 — Setup

In [ ]:
import os
import re
import sys
import dataclasses
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/during_imaging/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.common.io              import read_image_frames
from MERci.progress_display       import ProgressReporter
from MERci.visualization          import get_merci_figures_dir
from MERci.acquisition.configs    import find_frame_table_for_hal_config, get_all_color_frame_indices
from MERci.analysis.spot_localization       import detect_foci_per_z
from MERci.analysis.fast_spot_quantification import crop_center

print(f"SAMPLE_DIR: {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX  = ".zarr"   # must match what HAL is writing
NOTEBOOK_NAME = "z_profile_spot_intensity"   # namespaces this notebook's cache + figure files
MICROSCOPE    = "MF3"

HYB_ROUND_ID = 5     # which round's frame table to use for color -> z-frame-index / z(um) mapping,
                      # and (in standard mode) which round's SeriesInfo to resolve FOV_ID's file from
FOV_ID       = 12    # FOV to load, when not overridden by CUSTOM_IMAGE_PATH/CUSTOM_IMAGE_DIR below
CROP_SIZE    = None  # None = full frame; else a center crop in pixels (reuse fast_spot_quantification.crop_center)

# --- Custom path override (optional) --- leave both None to resolve the image the normal way
# (HYB_ROUND_ID's SeriesInfo + FOV_ID via round_info.csv/ExperimentMetadata).
CUSTOM_IMAGE_PATH = None   # literal path to one image file (.dax/.zarr/.tiff) -- read directly, bypassing
                           # ExperimentMetadata/SeriesInfo entirely (e.g. a one-off file outside data/)
CUSTOM_IMAGE_DIR  = None   # a directory outside round_info.csv's declared dirs (e.g. SAMPLE_DIR/"data/hybs/H01_old") --
                           # FOV_ID's filename is built inside it using HYB_ROUND_ID's own per-FOV naming convention
                           # (see SeriesInfo.resolve_path / candidate_dirs override, below)

# Colors to exclude from HYB_ROUND_ID's own resolution (Section 4) -- same convention as
# fast_spot_quantification.ipynb: 405 nm is the cells/DAPI channel, 488 nm is HAL's
# bead/focus-lock reference channel (fixed bead z, never real tissue signal).
EXCLUDED_COLORS = [405.0, 488.0]

# Foci-detection parameters for detect_foci_per_z (Section 5) -- same convention/defaults
# as fast_spot_quantification.ipynb's FOCI_MIN_DIST_PX/FOCI_THRESH_SIGMA.
FOCI_MIN_DIST_PX  = 4
FOCI_THRESH_SIGMA = 3.0

# False (default): skip recomputing if this run's cache CSV (Section 5) already exists.
OVERWRITE_OUTPUT = False

# Shared plot font sizes (NOTEBOOK_GUIDELINES.md #5) -- reused by every plotting cell.
PLOT_TITLE_FONTSIZE  = 14
PLOT_LABEL_FONTSIZE  = 12
PLOT_TICK_FONTSIZE   = 10
PLOT_LEGEND_FONTSIZE = 10

config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
    microscope     = MICROSCOPE,
)
meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt,
                                config.data_dir, image_suffix=config.image_suffix)

# NOTEBOOK_GUIDELINES.md #2: cache under analysis/cache/<notebook_name>/.
CACHE_DIR = config.analysis_dir / "cache" / NOTEBOOK_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = get_merci_figures_dir(SAMPLE_DIR, "during_imaging", NOTEBOOK_NAME)

print(f"Sample name  : {SAMPLE_NAME}")
print(f"FOVs (total) : {meta.n_fovs}")
print(f"Rounds       : {sorted(meta.rounds)}")
print(f"Cache dir    : {CACHE_DIR}")
print(f"Figures dir  : {FIGURES_DIR}")

## 3 — Resolve the raw image path

Three ways to resolve which raw image file this notebook reads, in priority
order:

1. `CUSTOM_IMAGE_PATH` set -> read that file directly, full stop.
2. `CUSTOM_IMAGE_DIR` set -> reuse `HYB_ROUND_ID`'s own `SeriesInfo` object(s)
   (`meta.series_for_round`), but with `candidate_dirs` overridden to point
   at the custom directory (`dataclasses.replace`) instead of writing new
   filename-pattern logic -- this reuses `SeriesInfo.resolve_path`'s exact
   per-FOV filename convention (zero-padding, regex fallback scan).
3. Neither set -> resolve the normal way, exactly like
   `fast_spot_quantification.compute_fov_round_color_spots` does.

If more than one series/imaging_type resolves a file for this round, the
first existing one is used (mirrors how
`fast_spot_quantification.compute_fov_round_color_spots` picks
`existing[0]`).

`RUN_ID` identifies this specific run for caching (Section 5): `(round,
fov)` in standard mode, or a filesystem-safe tag derived from the custom
path/dir + fov in the two custom modes, since there's no natural `(round,
fov)` key once the path is fully custom.

In [ ]:
if CUSTOM_IMAGE_PATH is not None:
    IMAGE_PATH = Path(CUSTOM_IMAGE_PATH)
    if not IMAGE_PATH.exists():
        raise FileNotFoundError(f"CUSTOM_IMAGE_PATH does not exist: {IMAGE_PATH}")
    fs_safe = lambda s: re.sub(r"[^A-Za-z0-9_.-]+", "_", str(s))
    RUN_ID = f"custompath_{fs_safe(IMAGE_PATH.stem)}_fov{FOV_ID:04d}"

elif CUSTOM_IMAGE_DIR is not None:
    custom_dir = Path(CUSTOM_IMAGE_DIR)
    existing = []
    for s in meta.series_for_round(HYB_ROUND_ID):
        # Reuse this series' own per-FOV filename convention (zero-padding,
        # regex fallback scan) against the custom directory instead of
        # duplicating that logic -- see SeriesInfo.resolve_path/candidate_dirs.
        custom_series = dataclasses.replace(s, candidate_dirs=[custom_dir])
        p = custom_series.resolve_path(FOV_ID, config.image_suffix)
        if p.exists():
            existing.append(p)
    if not existing:
        raise FileNotFoundError(
            f"No file for FOV {FOV_ID} found under {custom_dir} using round "
            f"{HYB_ROUND_ID}'s series naming convention(s)."
        )
    IMAGE_PATH = existing[0]
    fs_safe = lambda s: re.sub(r"[^A-Za-z0-9_.-]+", "_", str(s))
    RUN_ID = f"customdir_{fs_safe(custom_dir.name)}_fov{FOV_ID:04d}"

else:
    series = meta.series_for_round(HYB_ROUND_ID)
    existing = [p for p in (s.resolve_path(FOV_ID, config.image_suffix) for s in series) if p.exists()]
    if not existing:
        raise FileNotFoundError(f"FOV {FOV_ID} not yet imaged for round {HYB_ROUND_ID}.")
    IMAGE_PATH = existing[0]
    RUN_ID = f"round{HYB_ROUND_ID:03d}_fov{FOV_ID:04d}"

print(f"IMAGE_PATH: {IMAGE_PATH}")
print(f"RUN_ID    : {RUN_ID}")

## 4 — Resolve per-color frame indices and real z (µm)

Regardless of where the raw image itself came from (Section 3),
`HYB_ROUND_ID`'s own frame table always drives which camera frame indices
belong to which color and their real z (µm) -- same pattern as
`fast_spot_quantification.resolve_round_color_frame_indices`, except **every**
real z-plane is kept for each color (no `sample_z_frame_indices` picking),
since the whole point of this notebook is "every z", not a sample.

In [ ]:
COLOR_FRAME_INDICES = {}   # {color_nm: [frame_idx, ...]}  -- ascending real z, every plane
COLOR_Z_UM           = {}   # {color_nm: [z_um, ...]}        -- aligned 1:1 with COLOR_FRAME_INDICES

for s in meta.series_for_round(HYB_ROUND_ID):
    if not s.hal_config:
        continue
    frame_table_path = find_frame_table_for_hal_config(config.settings_dir / s.hal_config, config.metadata_dir)
    if frame_table_path is None:
        continue
    frame_table = pd.read_csv(frame_table_path)
    for color in sorted(frame_table["color"].dropna().unique()):
        if any(round(color) == round(excluded) for excluded in EXCLUDED_COLORS):
            continue
        all_indices = get_all_color_frame_indices(frame_table, float(color))
        if not all_indices:
            continue
        COLOR_FRAME_INDICES[float(color)] = all_indices
        COLOR_Z_UM[float(color)] = frame_table.loc[all_indices, "z"].values.astype(float)

print(f"Round {HYB_ROUND_ID} -- colors resolved: {sorted(COLOR_FRAME_INDICES)}")
for color_nm, idxs in COLOR_FRAME_INDICES.items():
    z_vals = COLOR_Z_UM[color_nm]
    print(f"  {color_nm:.0f} nm : {len(idxs)} real z-planes (z = {z_vals.min():.2f} .. {z_vals.max():.2f} um)")

## 5 — Per-z foci detection (cached)

For each color: read every real z-plane (Section 4's frame indices),
optionally center-crop, then run `detect_foci_per_z`
(`MERci.analysis.spot_localization`) -- the per-z (no max-projection)
counterpart to `fast_spot_quantification.detect_foci_in_crop` -- to get one
row per detected focus per z-plane. Cached under
`SAMPLE_DIR/analysis/cache/z_profile_spot_intensity/{RUN_ID}.csv` (skips
recomputation on a later run per NOTEBOOK_GUIDELINES.md #3, unless
`OVERWRITE_OUTPUT` is set).

In [ ]:
CACHE_PATH = CACHE_DIR / f"{RUN_ID}.csv"

if CACHE_PATH.exists() and not OVERWRITE_OUTPUT:
    SPOTS_DF = pd.read_csv(CACHE_PATH)
    print(f"Loaded cached result: {CACHE_PATH} ({len(SPOTS_DF)} rows)")
else:
    per_color_dfs = []
    reporter = ProgressReporter(total=len(COLOR_FRAME_INDICES), label="Detecting per-z foci by color")
    for color_nm in reporter.wrap(sorted(COLOR_FRAME_INDICES)):
        frame_indices = COLOR_FRAME_INDICES[color_nm]
        z_um_values   = COLOR_Z_UM[color_nm]
        stack = read_image_frames(IMAGE_PATH, frame_indices,
                                   frame_width=config.frame_width, frame_height=config.frame_height)
        if CROP_SIZE is not None:
            stack = np.stack([crop_center(f, CROP_SIZE) for f in stack], axis=0)
        color_df = detect_foci_per_z(stack, z_um_values, FOCI_MIN_DIST_PX, FOCI_THRESH_SIGMA)
        color_df["color_nm"] = color_nm
        per_color_dfs.append(color_df)

    SPOTS_DF = (pd.concat(per_color_dfs, ignore_index=True) if per_color_dfs
                else pd.DataFrame(columns=["z", "row_px", "col_px", "intensity", "color_nm"]))
    SPOTS_DF.to_csv(CACHE_PATH, index=False)
    print(f"Computed and cached: {CACHE_PATH} ({len(SPOTS_DF)} rows)")

## 6 — Plot: per-z spot intensity by color channel

One subplot per color channel: every detected focus's background-subtracted
intensity plotted against its real z (µm), plus a per-z median/IQR band --
a genuine 3-D punctum should show a smooth intensity peak across a few
z-planes; noise/dirt looks more like scattered single-plane blips.

In [ ]:
colors_present = sorted(SPOTS_DF["color_nm"].unique()) if not SPOTS_DF.empty else []

if not colors_present:
    print("No foci detected in any color/z-plane -- nothing to plot.")
else:
    fig, axes = plt.subplots(len(colors_present), 1, figsize=(10, 4 * len(colors_present)), squeeze=False)

    for ax, color_nm in zip(axes[:, 0], colors_present):
        sub = SPOTS_DF[SPOTS_DF["color_nm"] == color_nm]
        ax.scatter(sub["z"], sub["intensity"], s=10, alpha=0.35, color="steelblue", label="detected foci")

        stats = sub.groupby("z")["intensity"].agg(
            median="median",
            q25=lambda s: s.quantile(0.25),
            q75=lambda s: s.quantile(0.75),
        ).sort_index()
        ax.plot(stats.index, stats["median"], "-o", ms=3, color="firebrick", label="median")
        ax.fill_between(stats.index, stats["q25"], stats["q75"], color="firebrick", alpha=0.2, label="IQR")

        n_zplanes = sub["z"].nunique()
        ax.set_title(f"{color_nm:.0f} nm -- {len(sub)} detected foci across {n_zplanes} z-planes",
                     fontsize=PLOT_TITLE_FONTSIZE)
        ax.set_xlabel("z (µm)", fontsize=PLOT_LABEL_FONTSIZE)
        ax.set_ylabel("Spot intensity (bg-subtracted)", fontsize=PLOT_LABEL_FONTSIZE)
        ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
        ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)

    fig.suptitle(f"Per-z spot intensity -- {RUN_ID}", fontsize=PLOT_TITLE_FONTSIZE)
    fig.tight_layout()
    fig_path = FIGURES_DIR / f"{NOTEBOOK_NAME}.{RUN_ID}.z_profile.png"
    fig.savefig(fig_path, dpi=150)
    plt.show()
    print(f"Saved: {fig_path}")